# Addestramento

Questo notebook addestra la rete su un fold della validazione a finestra mobile e ne
misura la qualita' contro la persistenza.

Prerequisiti: i GRIB scaricati con `scripts/download_era5.py` e ingeriti con
`scripts/ingest_era5.py`. La cella di stato piu' sotto dice se ci sono dati a
sufficienza.

**Perche' a fold e non con una singola divisione.** Il periodo disponibile copre poco
piu' di due anni. Una divisione cronologica unica metterebbe nel test una sola
stagione, e il punteggio direbbe soprattutto in che mesi e' caduto il test. La
finestra mobile produce sei fold i cui blocchi di test coprono tutti i dodici mesi.

In [ ]:
import sys
from pathlib import Path

# Il notebook puo' essere aperto dalla cartella `notebooks/`: senza questo, l'import
# di `dwf` fallisce a seconda di dove e' stato avviato Jupyter.
RADICE = Path.cwd()
if not (RADICE / "src").exists():
    RADICE = RADICE.parent
sys.path.insert(0, str(RADICE / "src"))

from dwf.config import Config

config = Config.load(RADICE / "configs" / "default.yaml", project_root=RADICE)
print(f"periodo   : {config.time.start} .. {config.time.end}")
print(f"dominio   : {config.region.n_lat} x {config.region.n_lon}")
print(f"finestre  : {config.windows.input_slots} slot in ingresso -> "
      f"{config.windows.output_slots} previsti")
print(f"dati      : {config.paths.data_root}")

## 1. Stato dei dati

Ogni fold ha bisogno di finestre ammesse sia in train sia in validazione. Una finestra
e' ammessa solo se tutti i suoi 30 slot sono stati ingeriti e non contengono valori non
finiti: bastano pochi mesi mancanti per lasciare un fold senza dati.

In [ ]:
import polars as pl

from dwf.data.dataset import sample_starts

righe = []
for indice in range(len(config.build_folds())):
    righe.append({
        "fold": indice,
        "train": len(sample_starts(config, indice, "train")),
        "val": len(sample_starts(config, indice, "val")),
        "test": len(sample_starts(config, indice, "test")),
    })
disponibilita = pl.DataFrame(righe).with_columns(
    addestrabile=(pl.col("train") > 0) & (pl.col("val") > 0)
)
disponibilita

In [ ]:
addestrabili = disponibilita.filter("addestrabile").get_column("fold").to_list()
if not addestrabili:
    raise RuntimeError(
        "Nessun fold ha insieme finestre di train e di validazione: "
        "scaricare e ingerire altri mesi prima di proseguire."
    )
FOLD = addestrabili[0]
print(f"fold usato in questo notebook: {FOLD}")

## 2. Che cosa entra nella rete

L'ordine dei canali e' un contratto: scambiarne due non fa fallire nulla e produce solo
associazioni sbagliate. E' dichiarato una volta in `dwf.data.features` e tutto il resto
lo legge da li'.

In [ ]:
from dwf.data.features import InputLayout
from dwf.models.heads import OutputLayout

input_layout = InputLayout.from_config(config)
output_layout = OutputLayout.from_targets(config.targets, config.windows.output_slots)

print(f"canali in ingresso: {input_layout.n_channels}")
print(f"canali in uscita  : {output_layout.total_channels}")
input_layout.to_table().group_by("group").len().sort("group")

In [ ]:
# Le teste probabilistiche: la rete non prevede un numero ma una distribuzione, ed e'
# questo che rende misurabile l'affidabilita'.
pl.DataFrame(output_layout.describe())

## 3. Addestramento

La normalizzazione viene calcolata **solo** sugli slot di train del fold. Calcolarla su
tutti i dati farebbe entrare nel modello informazione dal futuro e la validazione
perderebbe significato.

Il costo misurato su questa macchina e' di circa 0,6 s per campione su ritagli 96x96.
Conviene partire con poche epoche per verificare che la perdita scenda, e allungare poi.

In [ ]:
from dwf.train import train_fold

EPOCHE = 5  # alzare dopo aver verificato che la perdita scende

esito = train_fold(config, FOLD, epochs=EPOCHE)
print(f"\nmigliore epoca: {esito.best_epoch}, validazione {esito.best_val_loss:.4f}")
print(f"checkpoint: {esito.checkpoint}")

In [ ]:
import matplotlib.pyplot as plt

epoche = [record.epoch for record in esito.history]
figura, assi = plt.subplots(1, 2, figsize=(12, 4))

assi[0].plot(epoche, [r.train_loss for r in esito.history], label="train")
assi[0].plot(epoche, [r.val_loss for r in esito.history], label="validazione")
assi[0].set_xlabel("epoca"); assi[0].set_ylabel("perdita"); assi[0].legend()
assi[0].set_title("Perdita totale")

# Le componenti separate dicono quale testa non sta imparando: senza, un miglioramento
# della temperatura potrebbe mascherare una precipitazione ferma.
for nome in esito.history[-1].components:
    assi[1].plot(epoche, [r.components.get(nome, float("nan")) for r in esito.history],
                 label=nome)
assi[1].set_xlabel("epoca"); assi[1].legend(); assi[1].set_title("Componenti (train)")
plt.tight_layout(); plt.show()

## 4. Il modello batte una previsione banale?

Un errore assoluto non dice nulla da solo: 2 K possono essere ottimi a tre giorni e
pessimi a sei ore. Il confronto con la persistenza, cioe' "domani come oggi", e' il
minimo sindacale.

In [ ]:
from dwf.data.dataset import WeatherWindowDataset, build_reader
from dwf.evaluate import collect_predictions, metrics_table, persistence_baseline
from dwf.train import load_checkpoint

rete, stats, input_layout, output_layout = load_checkpoint(config, FOLD)
lettore = build_reader(config, input_layout)
finestre_val = sample_starts(config, FOLD, "val")

dataset_val = WeatherWindowDataset(
    config, input_layout, stats, finestre_val, lettore,
    crop_size=None, crops_per_window=1, seed=config.training.seed,
)

MAX_FINESTRE = 8  # alzare per una stima piu' stabile, al costo di tempo
previsioni = collect_predictions(rete, dataset_val, output_layout, config,
                                 max_windows=MAX_FINESTRE)
riferimento = persistence_baseline(dataset_val, max_windows=MAX_FINESTRE)

metriche = pl.concat([
    metrics_table(previsioni, stats, model="dwf", split="val", fold=FOLD),
    metrics_table(riferimento, stats, model="persistence", split="val", fold=FOLD),
])
confronto = (
    metriche.filter((pl.col("lead_slot") >= 0) & (pl.col("month") == -1))
    .pivot(on="model", index=["variable", "metric", "lead_slot"], values="value")
    .sort("variable", "metric", "lead_slot")
)
confronto

In [ ]:
rmse = confronto.filter((pl.col("variable") == "t2m") & (pl.col("metric") == "rmse_celsius"))
plt.figure(figsize=(7, 4))
plt.plot(rmse["lead_slot"], rmse["dwf"], marker="o", label="modello")
plt.plot(rmse["lead_slot"], rmse["persistence"], marker="s", label="persistenza")
plt.xlabel("scadenza (slot di 6 ore)"); plt.ylabel("RMSE [K]")
plt.title("Temperatura a 2 m: modello contro persistenza")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5. Affidabilita'

Se il modello dice 30 % di pioggia, deve piovere nel 30 % dei casi in cui lo dice. Il
diagramma di affidabilita' verifica esattamente questo: i punti sulla diagonale sono
previsioni calibrate, sotto la diagonale sono eccessi di sicurezza.

In [ ]:
from dwf.evaluate import reliability_table

affidabilita = reliability_table(
    previsioni.tp_probability, previsioni.tp_occurrence,
    model="dwf", split="val", fold=FOLD, variable="tp",
)
validi = affidabilita.filter(pl.col("count") > 20)

plt.figure(figsize=(5.5, 5.5))
plt.plot([0, 1], [0, 1], "k--", label="calibrazione perfetta")
plt.plot(validi["forecast_mean"], validi["observed_frequency"], marker="o", label="modello")
plt.xlabel("probabilita' prevista"); plt.ylabel("frequenza osservata")
plt.title("Affidabilita' della probabilita' di pioggia")
plt.legend(); plt.grid(alpha=0.3); plt.show()
validi

## 6. Prossimi passi

- Alzare `EPOCHE` finche' la validazione smette di migliorare.
- Addestrare gli altri fold: `for fold in addestrabili: train_fold(config, fold)`.
- Ingerire altri mesi per riempire i fold ancora vuoti e coprire tutte le stagioni.